In [6]:
!pip install -q -U transformers accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 92.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 64.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

In [7]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd

import torch
import wandb

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

warnings.filterwarnings("ignore")

In [8]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
Tesla T4


In [9]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SUB_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SUB_PATH)

print(train_df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [10]:
def create_prompt(df):
    return (
        "Question: " + df["prompt"].fillna("") + "\n\n"
        "Option A: " + df["A"].fillna("") + "\n"
        "Option B: " + df["B"].fillna("") + "\n"
        "Option C: " + df["C"].fillna("") + "\n"
        "Option D: " + df["D"].fillna("") + "\n"
        "Option E: " + df["E"].fillna("")
    )

train_df["text"] = create_prompt(train_df)
test_df["text"] = create_prompt(test_df)

In [11]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

train_df["label"] = train_df["answer"].map(label2id)

In [13]:
train_split, valid_split = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df["label"],
    random_state=42
)

In [14]:
train_dataset = Dataset.from_pandas(train_split)
valid_dataset = Dataset.from_pandas(valid_split)
test_dataset = Dataset.from_pandas(test_df)

In [15]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ Tokenizer Loaded!")
print(tokenizer)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

✅ Tokenizer Loaded!
DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [16]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.de

In [17]:
MAX_LENGTH = 384

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [18]:
keep_train = ["input_ids", "attention_mask", "label"]
keep_test = ["input_ids", "attention_mask"]

train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c not in keep_train]
)

valid_dataset = valid_dataset.remove_columns(
    [c for c in valid_dataset.column_names if c not in keep_train]
)

test_dataset = test_dataset.remove_columns(
    [c for c in test_dataset.column_names if c not in keep_test]
)

In [19]:
print(train_dataset)

print(train_dataset[0])

print("Max Token ID:", max(train_dataset[0]["input_ids"]))
print("Tokenizer Vocab:", tokenizer.vocab_size)

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 1800
})
{'label': 4, 'input_ids': [1, 7581, 294, 458, 269, 262, 2193, 810, 265, 33021, 270, 262, 3546, 280, 268, 1664, 271, 268, 13092, 104741, 302, 12395, 336, 294, 60245, 12395, 736, 294, 112246, 326, 12395, 716, 294, 36522, 33021, 12395, 802, 294, 112246, 2231, 12395, 829, 294, 36272, 265, 262, 33021, 537, 287, 25687, 285, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Max Token ID: 112246
Tokenizer Vocab: 128000


In [21]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [23]:
import numpy as np

def map3_score(logits, labels):
    top3 = np.argsort(logits, axis=1)[:, -3:][:, ::-1]

    score = 0

    for pred, true in zip(top3, labels):
        if true in pred:
            score += 1 / (np.where(pred == true)[0][0] + 1)

    return score / len(labels)

In [24]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    map3 = map3_score(logits, labels)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "map3": map3,
    }

In [25]:
training_args = TrainingArguments(
    output_dir="./deberta-v3-small",

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    learning_rate=1e-5,          

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    num_train_epochs=5,

    weight_decay=0.01,

    warmup_steps=100,           

    max_grad_norm=1.0,           

    fp16=False,                  
    bf16=True,

    load_best_model_at_end=True,

    metric_for_best_model="map3",
    greater_is_better=True,

    report_to="none",

    save_total_limit=1,

    seed=42,
)

In [28]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stopping]
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Map3
1,6.475662,2.620610,0.380000,0.303367,0.380000,0.259271,0.612500
2,5.677758,3.366334,0.185000,0.034225,0.185000,0.057764,0.353333
3,20.580103,nan,0.185000,0.034225,0.185000,0.057764,0.326667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=171, training_loss=9.571205362241868, metrics={'train_runtime': 76.1721, 'train_samples_per_second': 118.154, 'train_steps_per_second': 3.742, 'total_flos': 508326847073760.0, 'train_loss': 9.571205362241868, 'epoch': 3.0})

In [31]:
trainer.save_model("./deberta-v3-small-finetuned")
tokenizer.save_pretrained("./deberta-v3-small-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./deberta-v3-small-finetuned/tokenizer_config.json',
 './deberta-v3-small-finetuned/tokenizer.json')

In [32]:
ROBERTA_MODEL = "FacebookAI/roberta-base"

In [33]:
from transformers import AutoTokenizer

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

print("✅ RoBERTa Tokenizer Loaded!")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ RoBERTa Tokenizer Loaded!


In [34]:
from transformers import AutoModelForSequenceClassification

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("✅ RoBERTa Model Loaded!")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ RoBERTa Model Loaded!


In [36]:
from datasets import Dataset

roberta_train_dataset = Dataset.from_pandas(train_split)
roberta_valid_dataset = Dataset.from_pandas(valid_split)
roberta_test_dataset = Dataset.from_pandas(test_df)

In [37]:
MAX_LENGTH = 384

def roberta_tokenize(batch):
    return roberta_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

roberta_train_dataset = roberta_train_dataset.map(
    roberta_tokenize,
    batched=True
)

roberta_valid_dataset = roberta_valid_dataset.map(
    roberta_tokenize,
    batched=True
)

roberta_test_dataset = roberta_test_dataset.map(
    roberta_tokenize,
    batched=True
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [38]:
keep_train = ["input_ids", "attention_mask", "label"]
keep_test = ["input_ids", "attention_mask"]

roberta_train_dataset = roberta_train_dataset.remove_columns(
    [c for c in roberta_train_dataset.column_names if c not in keep_train]
)

roberta_valid_dataset = roberta_valid_dataset.remove_columns(
    [c for c in roberta_valid_dataset.column_names if c not in keep_train]
)

roberta_test_dataset = roberta_test_dataset.remove_columns(
    [c for c in roberta_test_dataset.column_names if c not in keep_test]
)

print(roberta_train_dataset.column_names)

['label', 'input_ids', 'attention_mask']


In [39]:
from transformers import DataCollatorWithPadding

roberta_data_collator = DataCollatorWithPadding(
    tokenizer=roberta_tokenizer
)

In [40]:
roberta_training_args = TrainingArguments(

    output_dir="./roberta-base",


    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    num_train_epochs=5,

    weight_decay=0.01,
    warmup_steps=100,

    fp16=False,
    bf16=True,

    load_best_model_at_end=True,

    metric_for_best_model="map3",
    greater_is_better=True,

    report_to="none",

    save_total_limit=1,

    seed=42,
)

In [41]:
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,

    train_dataset=roberta_train_dataset,
    eval_dataset=roberta_valid_dataset,
    data_collator=roberta_data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stopping]
)

In [42]:
roberta_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Map3
1,6.433779,3.175003,0.295000,0.135807,0.295000,0.183728,0.446667
2,5.706083,1.065861,0.900000,0.912233,0.900000,0.899232,0.942500
3,1.646537,0.026624,1.000000,1.000000,1.000000,1.000000,1.000000
4,0.117482,0.003822,1.000000,1.000000,1.000000,1.000000,1.000000
5,0.013389,0.002974,1.000000,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=285, training_loss=2.4428096464851445, metrics={'train_runtime': 409.2104, 'train_samples_per_second': 21.994, 'train_steps_per_second': 0.696, 'total_flos': 1715337061479408.0, 'train_loss': 2.4428096464851445, 'epoch': 5.0})

In [43]:
roberta_trainer.save_model("./roberta-base-finetuned")
roberta_tokenizer.save_pretrained("./roberta-base-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./roberta-base-finetuned/tokenizer_config.json',
 './roberta-base-finetuned/tokenizer.json')

In [44]:
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForSequenceClassification

deberta_tokenizer = AutoTokenizer.from_pretrained(
    "./deberta-v3-small-finetuned"
)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "./deberta-v3-small-finetuned"
)

deberta_model.eval()

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [45]:
sample = train_df.iloc[25]

text = sample["text"]

print(text)

Question: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully.

Option A: Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.
Option B: Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.
Option C: Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.
Option D: Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.
Option E: Hesse's principle of transfer is a concept in geometry that states that if the points of the projective line P1 are depicted by a rational normal curve in Pn, then the group of the projective transformations of Pn that preserve the curve is isomorphic to the group of the projective transformations of P1.


In [46]:
inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=384
)

In [47]:
with torch.no_grad():
    outputs = deberta_model(**inputs)

logits = outputs.logits

probabilities = F.softmax(logits, dim=1)

print(probabilities)

tensor([[0.0725, 0.4153, 0.2091, 0.1998, 0.1033]], dtype=torch.float16)


In [48]:
options = ["A", "B", "C", "D", "E"]

pred = torch.argmax(probabilities, dim=1).item()

print("Predicted Option:", options[pred])
print("Probability:", probabilities[0][pred].item())

Predicted Option: B
Probability: 0.415283203125


In [50]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

roberta_model.to(device)

sample = train_df.iloc[25]
text = sample["text"]

inputs = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=384
)

# Move every tensor to the GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = roberta_model(**inputs)

roberta_probabilities = F.softmax(outputs.logits, dim=1)

print(roberta_probabilities)

tensor([[0.0022, 0.0035, 0.0025, 0.0019, 0.9899]], device='cuda:0')


In [51]:
deberta_model.to(device)

sample = train_df.iloc[25]
text = sample["text"]

inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=384
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = deberta_model(**inputs)

deberta_probabilities = F.softmax(outputs.logits, dim=1)

print(deberta_probabilities)

tensor([[0.0725, 0.4153, 0.2091, 0.1998, 0.1033]], device='cuda:0',
       dtype=torch.float16)


In [52]:
average_probabilities = (
    deberta_probabilities.cpu() +
    roberta_probabilities.cpu()
) / 2

print(average_probabilities)

tensor([[0.0373, 0.2094, 0.1058, 0.1009, 0.5466]])


In [53]:
options = ["A", "B", "C", "D", "E"]

pred = average_probabilities.argmax(dim=1).item()

print("Predicted Option:", options[pred])
print("Average Probability:", average_probabilities[0][pred].item())

Predicted Option: E
Average Probability: 0.5465757250785828


In [54]:
weighted_probabilities = (
    0.7 * deberta_probabilities.cpu() +
    0.3 * roberta_probabilities.cpu()
)

print(weighted_probabilities)

tensor([[0.0514, 0.2918, 0.1471, 0.1405, 0.3692]])


In [55]:
options = ["A", "B", "C", "D", "E"]

pred = weighted_probabilities.argmax(dim=1).item()

print("Predicted Option:", options[pred])
print("Weighted Probability:", weighted_probabilities[0][pred].item())

Predicted Option: E
Weighted Probability: 0.36922961473464966


In [56]:
options = ["A", "B", "C", "D", "E"]

# Sort probabilities in descending order
top_indices = weighted_probabilities.argsort(dim=1, descending=True)[0]

# Convert indices to option labels
ranked_options = [options[i] for i in top_indices.tolist()]

print("Ranking:", ranked_options)
print("Top-3:", " ".join(ranked_options[:3]))

Ranking: ['E', 'B', 'C', 'D', 'A']
Top-3: E B C


In [57]:
print(len(test_df))

500


In [59]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
deberta_model.eval()

changed = 0

for i in range(50):

    original = test_df.iloc[i]["text"]

    augmented = (
        "Answer the following multiple-choice question carefully:\n\n"
        + original
    )

    # Original prompt
    inputs1 = deberta_tokenizer(
        original,
        return_tensors="pt",
        truncation=True,
        max_length=384,
    )
    inputs1 = {k: v.to(device) for k, v in inputs1.items()}

    with torch.no_grad():
        logits1 = deberta_model(**inputs1).logits

    prob1 = F.softmax(logits1, dim=1)

    # Augmented prompt
    inputs2 = deberta_tokenizer(
        augmented,
        return_tensors="pt",
        truncation=True,
        max_length=384,
    )
    inputs2 = {k: v.to(device) for k, v in inputs2.items()}

    with torch.no_grad():
        logits2 = deberta_model(**inputs2).logits

    prob2 = F.softmax(logits2, dim=1)

    # Average probabilities
    avg_prob = (prob1 + prob2) / 2

    # Compare Top-1 predictions
    pred_original = prob1.argmax(dim=1).item()
    pred_avg = avg_prob.argmax(dim=1).item()

    if pred_original != pred_avg:
        changed += 1

print("Number of changed predictions:", changed)

Number of changed predictions: 0


In [62]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model.eval()
roberta_model.eval()

changed = 0

for i in range(100):

    text = test_df.iloc[i]["text"]

    # ---------------- DeBERTa ----------------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=1)

    # ---------------- RoBERTa ----------------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=1)

    # ---------------- Weighted Ensemble ----------------
    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_pred = deberta_probs.argmax(dim=1).item()
    ensemble_pred = weighted_probs.argmax(dim=1).item()

    if deberta_pred != ensemble_pred:
        changed += 1

print("Different Top-1 Predictions:", changed)

Different Top-1 Predictions: 55


In [63]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model.eval()
roberta_model.eval()

positive_gain = 0

for i in range(100):

    text = test_df.iloc[i]["text"]

    # ---------------- DeBERTa ----------------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=1)

    # ---------------- RoBERTa ----------------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=1)

    # ---------------- Weighted Ensemble ----------------
    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_conf = deberta_probs.max().item()
    ensemble_conf = weighted_probs.max().item()

    confidence_gain = ensemble_conf - deberta_conf

    if confidence_gain > 0:
        positive_gain += 1

print("Positive Confidence Gain:", positive_gain)

Positive Confidence Gain: 80


In [64]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model.eval()
roberta_model.eval()

options = ["A", "B", "C", "D", "E"]

changed = 0

for i in range(100):

    text = test_df.iloc[i]["text"]

    # ---------- DeBERTa ----------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=1)

    # ---------- RoBERTa ----------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=384
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=1)

    # ---------- Weighted Ensemble ----------
    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    # Top-3 rankings
    deberta_top3 = torch.argsort(deberta_probs, dim=1, descending=True)[0][:3].tolist()
    ensemble_top3 = torch.argsort(weighted_probs, dim=1, descending=True)[0][:3].tolist()

    if deberta_top3 != ensemble_top3:
        changed += 1

print("Rows with changed Top-3 ranking:", changed)

Rows with changed Top-3 ranking: 60


In [67]:
import torch
import torch.nn.functional as F
import numpy as np

deberta_model.eval()
roberta_model.eval()

all_probs = []
all_labels = []

for i in range(100):

    sample = valid_dataset[i]

    # DeBERTa
    inputs = {
        "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0).to(device),
        "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0).to(device),
    }

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=1)

    # RoBERTa
    inputs = {
        "input_ids": torch.tensor(roberta_valid_dataset[i]["input_ids"]).unsqueeze(0).to(device),
        "attention_mask": torch.tensor(roberta_valid_dataset[i]["attention_mask"]).unsqueeze(0).to(device),
    }

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=1)

    weighted = 0.7 * deberta_probs + 0.3 * roberta_probs

    all_probs.append(weighted.cpu().numpy()[0])
    all_labels.append(sample["label"])

In [68]:
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(map3_score(all_probs, all_labels))

0.98
